# BdSL → Bangla FIR LLM — Exploration Notebook

This notebook walks through:
1. Dataset generation & inspection
2. Prompt engineering
3. Quick training run (small subset)
4. Inference & evaluation demo

**Prerequisites**: run `pip install -r requirements.txt` from the `llm/` directory.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import json
import pandas as pd
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

## 1. Generate Synthetic Dataset

In [ ]:
from data.synthetic.fir_templates import generate_dataset, build_instruction_pairs

samples = generate_dataset(n_samples=500, seed=42)
pairs   = build_instruction_pairs(samples)

print(f'Total samples: {len(pairs)}')

# Class distribution
dist = Counter(p['offense_type'] for p in pairs)
print('\nOffense distribution:')
for k, v in dist.most_common():
    print(f'  {k:20s}: {v}')

In [ ]:
# Inspect a sample
sample = pairs[0]
print('=== INSTRUCTION ===')
print(sample['instruction'])
print('\n=== INFORMAL INPUT ===')
print(sample['input'])
print('\n=== FORMAL FIR OUTPUT ===')
print(sample['output'])

In [ ]:
# DataFrame view
df = pd.DataFrame([
    {'offense': p['offense_type'], 'input_len': len(p['input']), 'output_len': len(p['output'])}
    for p in pairs
])
df.describe()

## 2. Prompt Template Inspection

In [ ]:
from src.data_prep import format_prompt, DEFAULT_INSTRUCTION

example_text = 'গতকাল রাত ১০টায় মিরপুর ১০ এলাকায় বাদল মিয়া আমার মোবাইল ছিনিয়ে নিয়েছে।'

print('=== Inference Prompt (no output) ===')
print(format_prompt(DEFAULT_INSTRUCTION, example_text))

## 3. Entity Extraction Demo

In [ ]:
from src.inference import extract_entities

sample_fir = pairs[0]['output']
entities = extract_entities(sample_fir)

print('Extracted entities:')
for field, value in entities.__dict__.items():
    print(f'  {field:20s}: {value}')

## 4. Metrics Demo (on reference text)

In [ ]:
from src.utils import compute_bleu, compute_rouge

# Simulate perfect predictions for baseline check
refs  = [p['output'] for p in pairs[:10]]
preds = refs[:]   # perfect match

print('BLEU (perfect):', compute_bleu(preds, refs))
print('ROUGE (perfect):', compute_rouge(preds, refs))

## 5. Live Inference (requires fine-tuned model)

Uncomment and run after completing training.

In [ ]:
# from src.inference import FIRGenerator, InferenceConfig
#
# cfg = InferenceConfig(
#     model_path='../models/checkpoints/final_adapter',
#     base_model_name='unsloth/llama-3-8b-bnb-4bit',
# )
# gen = FIRGenerator(cfg)
#
# test_inputs = [
#     'গতকাল রাত ১০টায় মিরপুর ১০ এলাকায় বাদল মিয়া আমার মোবাইল ছিনিয়ে নিয়েছে।',
#     'উত্তরা ৫ নম্বর সেক্টরে সকাল ৮টায় রফিক উদ্দিন আমার মানিব্যাগ চুরি করে পালিয়ে গেছে।',
#     'কালাম শেখ আমাকে জীবননাশের হুমকি দিয়েছে। ঘটনা গুলশান ২ এলাকায় বিকাল ৫টায়।',
# ]
#
# for text in test_inputs:
#     print('INPUT:', text)
#     result = gen.generate_structured(text)
#     print('FIR:\n', result['fir_text'][:500], '...')
#     print('ENTITIES:', result['entities'])
#     print('---')

## 6. Save dataset for training

In [ ]:
import subprocess
result = subprocess.run(
    ['python', '../scripts/generate_data.py', '--n_samples', '3000'],
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr)